# Análisis de sentimiento con una red LSTM

**Dataset:** IMDB Movie Reviews · **Framework:** TensorFlow / Keras · **Entorno:** Google Colab

El objetivo es entrenar una red neuronal recurrente que lea una reseña de película y diga si la opinión es positiva o negativa. Al final el modelo queda guardado y lo usa una aplicación en Streamlit donde cualquiera puede escribir su propia reseña.

Antes de correr: en Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

## 1. Librerías

In [ ]:
import json
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score)

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 2. Selección y descripción del dataset

Usé el **IMDB Large Movie Review Dataset**, publicado en 2011 por Andrew Maas y su equipo en la Universidad de Stanford. Son 50.000 reseñas de películas escritas por usuarios de IMDB, en inglés, cada una marcada como positiva o negativa.

Lo elegí por tres razones. Es texto real, con errores, ironía y opiniones mezcladas, no frases armadas para el ejercicio. Las dos clases están balanceadas, 25.000 y 25.000, así que el accuracy se puede leer sin trampas. Y viene incluido en Keras, ya separado en entrenamiento y prueba.

| Característica | Valor |
|---|---|
| Reseñas totales | 50.000 (25.000 entrenamiento + 25.000 prueba) |
| Idioma | Inglés |
| Clases | 2: negativa (0) y positiva (1) |
| Criterio de las etiquetas | Positiva si el usuario dio 7 o más estrellas de 10; negativa si dio 4 o menos |
| Tipo de problema | Clasificación binaria de secuencias |

Las reseñas con 5 o 6 estrellas no están en el dataset. Eso deja afuera las opiniones tibias y hace el problema un poco más fácil de lo que sería en la vida real.

Keras entrega cada reseña ya convertida en una lista de números: cada número representa una palabra. Me quedo con las **10.000 palabras más frecuentes**; las demás se reemplazan por una marca de "palabra desconocida".

In [ ]:
NUM_PALABRAS = 10000
CLASES = ["negativa", "positiva"]

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=NUM_PALABRAS)

print("Reseñas de entrenamiento:", len(x_train_full))
print("Reseñas de prueba:       ", len(x_test))
print("\nAsí se ve la primera reseña por dentro (primeros 20 números):")
print(x_train_full[0][:20])
print("Etiqueta:", y_train_full[0], "→", CLASES[y_train_full[0]])

Para leer las reseñas como texto hace falta el diccionario que traduce número a palabra. Keras reserva tres números: **0** para relleno, **1** para marcar el inicio de la reseña y **2** para las palabras desconocidas. Por eso los índices del diccionario se corren 3 posiciones.

In [ ]:
indice_palabras = keras.datasets.imdb.get_word_index()

# palabra -> número, tal como aparece en las secuencias (corrido 3 posiciones)
palabra_a_numero = {palabra: i + 3 for palabra, i in indice_palabras.items()}
numero_a_palabra = {i: palabra for palabra, i in palabra_a_numero.items()}
numero_a_palabra.update({0: "<pad>", 1: "<inicio>", 2: "<?>"})


def decodificar(secuencia):
    return " ".join(numero_a_palabra.get(i, "<?>") for i in secuencia if i not in (0, 1))


print("Palabras en el diccionario completo:", len(indice_palabras))
print()
print("Reseña 0:", decodificar(x_train_full[0])[:500], "...")

## 3. Análisis exploratorio de los datos

### 3.1 ¿Cuántas reseñas hay de cada clase?

In [ ]:
conteo = pd.Series(y_train_full).value_counts().sort_index()
conteo.index = CLASES

plt.figure(figsize=(5, 4))
sns.barplot(x=conteo.index, y=conteo.values, hue=conteo.index, palette="Set2", legend=False)
plt.title("Reseñas por clase (entrenamiento)")
plt.xlabel("Sentimiento")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.savefig("reports/resenas_por_clase.png", dpi=120)
plt.show()

print(conteo)

### 3.2 ¿Qué tan largas son las reseñas?

Este dato decide un parámetro importante. La red necesita que todas las secuencias tengan el mismo largo, así que hay que elegir un número de palabras y recortar o rellenar las reseñas para llegar a él.

In [ ]:
longitudes = pd.Series([len(r) for r in x_train_full])

plt.figure(figsize=(10, 4))
plt.hist(longitudes, bins=60, color="steelblue", edgecolor="white")
plt.axvline(200, color="red", linestyle="--", label="200 palabras")
plt.title("Largo de las reseñas (entrenamiento)")
plt.xlabel("Número de palabras")
plt.ylabel("Cantidad de reseñas")
plt.legend()
plt.tight_layout()
plt.savefig("reports/largo_resenas.png", dpi=120)
plt.show()

print(f"Más corta:  {longitudes.min()} palabras")
print(f"Más larga:  {longitudes.max()} palabras")
print(f"Mediana:    {longitudes.median():.0f} palabras")
print(f"Promedio:   {longitudes.mean():.0f} palabras")
print(f"Reseñas con 200 palabras o menos: {(longitudes <= 200).mean():.1%}")

In [ ]:
# ¿Las reseñas positivas son más largas o más cortas que las negativas?
largo_por_clase = pd.DataFrame({"largo": longitudes, "clase": [CLASES[y] for y in y_train_full]})
print(largo_por_clase.groupby("clase")["largo"].describe().round(0))

### 3.3 Palabras más frecuentes en cada clase

Quito las palabras vacías del inglés (the, and, a, of...), porque aparecen en todas las reseñas y no dicen nada del sentimiento.

In [ ]:
PALABRAS_VACIAS = set("""
the and a of to is in it i this that was as with for but on movie film his he you are
not be have one all at they by an who so from like her or just about out has if what
there some can more when very up no time even my would which only she their see story
than really had were me well much also do get been other its how into will because
we br don't it's any most made them him then could two make way too after first many
films movies people seen watch know does think did character characters
""".split())


def frecuentes(etiqueta, n=15):
    contador = Counter()
    for secuencia, y in zip(x_train_full, y_train_full):
        if y == etiqueta:
            contador.update(numero_a_palabra.get(i) for i in secuencia if i > 2)
    return [(p, c) for p, c in contador.most_common(300) if p not in PALABRAS_VACIAS][:n]


fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
for eje, etiqueta, color in zip(ejes, [0, 1], ["indianred", "seagreen"]):
    palabras, cantidades = zip(*frecuentes(etiqueta))
    eje.barh(palabras[::-1], cantidades[::-1], color=color)
    eje.set_title(f"Palabras más frecuentes en reseñas {CLASES[etiqueta]}s")
    eje.set_xlabel("Apariciones")
plt.tight_layout()
plt.savefig("reports/palabras_frecuentes.png", dpi=120)
plt.show()

Varias palabras aparecen en las dos listas, como "good" o "great". Una reseña negativa puede decir "not good" o "I expected a great film". Esa es justamente la razón para usar una red recurrente: necesita leer las palabras **en orden** para entender que "not" cambia el sentido de "good".

### 3.4 Ejemplos de reseñas

In [ ]:
for i in [1, 2, 5]:
    print(f"[{CLASES[y_train_full[i]].upper()}] {decodificar(x_train_full[i])[:400]}...")
    print()

## 4. Limpieza y preprocesamiento

El dataset de Keras ya viene limpio y tokenizado: minúsculas, sin puntuación, cada palabra convertida en número. Aun así quedan tres tareas.

**1. Vocabulario limitado.** Ya quedó en 10.000 palabras al cargar. Las palabras raras, que aparecen pocas veces, se cambian por el número 2. Con eso la red no gasta parámetros en palabras que vio una o dos veces.

**2. Largo fijo.** Uso 200 palabras. Las reseñas más largas se recortan quedándose con las **últimas** 200, porque el final de una reseña suele traer la conclusión ("in the end, I recommend it"). Las más cortas se rellenan con ceros **al principio**, así la LSTM termina de leer justo en la última palabra real.

**3. Limpieza de textos nuevos.** La aplicación va a recibir texto escrito por personas, con mayúsculas, signos y saltos de línea. Esta función lo deja igual al formato del dataset. Es la misma que usa `src/predict.py`.

In [ ]:
LONGITUD = 200
INICIO, DESCONOCIDA = 1, 2

# Solo las palabras que la red conoce (número menor a 10.000)
vocabulario = {p: n for p, n in palabra_a_numero.items() if n < NUM_PALABRAS}
print("Palabras en el vocabulario del modelo:", len(vocabulario))


def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"<br\s*/?>", " ", texto)
    texto = re.sub(r"[^a-z0-9'\s]", " ", texto)
    return texto.split()


def codificar(texto):
    palabras = limpiar_texto(texto)
    secuencia = [INICIO] + [vocabulario.get(p, DESCONOCIDA) for p in palabras]
    return keras.utils.pad_sequences([secuencia], maxlen=LONGITUD)


ejemplo = "This movie was NOT good!<br />The actors... terrible."
print("\nOriginal:", ejemplo)
print("Limpio:  ", limpiar_texto(ejemplo))
print("Números: ", codificar(ejemplo)[0][-10:], "(últimos 10 de 200)")

In [ ]:
x_train_full = keras.utils.pad_sequences(x_train_full, maxlen=LONGITUD)
x_test = keras.utils.pad_sequences(x_test, maxlen=LONGITUD)

print("Forma del entrenamiento:", x_train_full.shape)
print("Forma de la prueba:     ", x_test.shape)
print("\nUna reseña corta después de rellenar (primeros 20 valores):")
corta = int(np.argmin(longitudes))
print(x_train_full[corta][:20])

Los ceros del principio son el relleno. A diferencia de las imágenes, aquí no se normaliza dividiendo entre algo: los números son identificadores de palabras, no cantidades. La capa **Embedding** de la red se encarga de convertir cada identificador en un vector con significado.

## 5. División en entrenamiento, validación y prueba

Keras ya separa 25.000 reseñas de prueba, que no se tocan hasta el final. Del entrenamiento saco un 20% para validación, con `stratify` para mantener la mitad positiva y la mitad negativa.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.20,
    stratify=y_train_full, random_state=SEMILLA)

print(f"Entrenamiento: {len(x_train):>6} reseñas")
print(f"Validación:    {len(x_val):>6} reseñas")
print(f"Prueba:        {len(x_test):>6} reseñas")
print("\nProporción de positivas:")
print(f"  entrenamiento {y_train.mean():.3f} | validación {y_val.mean():.3f} | prueba {y_test.mean():.3f}")

## 6. Construcción de la red

Elegí una **LSTM** en vez de una RNN simple. Una RNN simple va olvidando lo que leyó al principio a medida que avanza: con 200 palabras, cuando llega al final casi no recuerda el comienzo. Es el problema del desvanecimiento del gradiente. La LSTM tiene "compuertas" que deciden qué guardar y qué olvidar, y aguanta secuencias mucho más largas.

Las capas:

- **Embedding (10.000 → 64):** convierte cada palabra en un vector de 64 números. La red aprende estos vectores, y palabras que se usan parecido, como "great" y "excellent", terminan con vectores cercanos.
- **Bidirectional(LSTM(64)):** dos LSTM, una lee la reseña de izquierda a derecha y otra de derecha a izquierda. Así cada palabra se interpreta con lo que viene antes y con lo que viene después.
- **Dropout (50% y 30%):** apaga neuronas al azar durante el entrenamiento para que la red no memorice reseñas.
- **Dense final con sigmoid:** una sola salida entre 0 y 1. Es la probabilidad de que la reseña sea positiva.

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(LONGITUD,)),
    layers.Embedding(input_dim=NUM_PALABRAS, output_dim=64),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.5),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="lstm_imdb")

modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
               loss="binary_crossentropy",
               metrics=["accuracy"])

modelo.summary()

Uso `binary_crossentropy` porque hay solo dos clases y una salida. Fíjate en el resumen: la capa Embedding sola tiene 640.000 parámetros (10.000 palabras × 64), casi todo el modelo.

## 7. Entrenamiento

Pongo hasta 10 épocas con **EarlyStopping**: si la pérdida en validación no mejora en 2 épocas seguidas, corta y recupera los pesos de la mejor época. En texto el sobreajuste llega rápido, muchas veces desde la época 3 o 4.

Con GPU T4 cada época tarda unos 15 a 30 segundos.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
]

historial = modelo.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
)

print("Épocas entrenadas:", len(historial.history["loss"]))

## 8. Gráficas del entrenamiento

In [ ]:
h = historial.history
epocas = range(1, len(h["loss"]) + 1)
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))

ejes[0].plot(epocas, h["accuracy"], marker="o", label="Entrenamiento")
ejes[0].plot(epocas, h["val_accuracy"], marker="o", label="Validación")
ejes[0].set_title("Accuracy por época")
ejes[0].set_xlabel("Época")
ejes[0].set_ylabel("Accuracy")
ejes[0].legend()

ejes[1].plot(epocas, h["loss"], marker="o", label="Entrenamiento")
ejes[1].plot(epocas, h["val_loss"], marker="o", label="Validación")
ejes[1].set_title("Pérdida (loss) por época")
ejes[1].set_xlabel("Época")
ejes[1].set_ylabel("Loss")
ejes[1].legend()

plt.tight_layout()
plt.savefig("reports/entrenamiento.png", dpi=120)
plt.show()

mejor = int(np.argmin(h["val_loss"]))
print(f"Mejor época: {mejor + 1}")
print(f"  Accuracy entrenamiento: {h['accuracy'][mejor]:.4f}")
print(f"  Accuracy validación:    {h['val_accuracy'][mejor]:.4f}")
print(f"  Loss validación:        {h['val_loss'][mejor]:.4f}")

**Cómo leer las gráficas:** si las curvas de entrenamiento y validación van juntas, el modelo generaliza. Si la de entrenamiento sigue mejorando mientras la de validación se estanca o empeora, empezó a memorizar.

[Escribe aquí lo que ves al correrlo: en qué época se detuvo, cuál fue la mejor y si las curvas se separan.]

## 9. Evaluación en el conjunto de prueba

Además del accuracy uso:

- **Precision:** de las reseñas que el modelo llamó positivas, cuántas lo eran.
- **Recall:** de todas las reseñas positivas reales, cuántas encontró.
- **F1:** el balance entre las dos.
- **AUC (área bajo la curva ROC):** mide qué tan bien separa las clases sin depender del umbral de 0.5. Un 0.5 es azar y un 1.0 es perfecto.

In [ ]:
perdida_test, accuracy_test = modelo.evaluate(x_test, y_test, verbose=0)
probabilidades = modelo.predict(x_test, verbose=0).ravel()
y_pred = (probabilidades >= 0.5).astype(int)
auc_test = roc_auc_score(y_test, probabilidades)

print(f"Accuracy:  {accuracy_test:.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"AUC:       {auc_test:.4f}")
print(f"Pérdida:   {perdida_test:.4f}")

reporte = classification_report(y_test, y_pred, target_names=CLASES, digits=3)
print()
print(reporte)

with open("reports/metricas.txt", "w", encoding="utf-8") as f:
    f.write(f"Accuracy prueba: {accuracy_test:.4f}\nPérdida prueba: {perdida_test:.4f}\n"
            f"AUC: {auc_test:.4f}\n\n{reporte}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
matriz = confusion_matrix(y_test, y_pred)
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASES, yticklabels=CLASES, ax=ejes[0])
ejes[0].set_xlabel("Clase predicha")
ejes[0].set_ylabel("Clase real")
ejes[0].set_title("Matriz de confusión (25.000 reseñas de prueba)")

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, probabilidades)
ejes[1].plot(fpr, tpr, label=f"LSTM (AUC = {auc_test:.3f})")
ejes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Azar (AUC = 0.5)")
ejes[1].set_xlabel("Tasa de falsos positivos")
ejes[1].set_ylabel("Tasa de verdaderos positivos")
ejes[1].set_title("Curva ROC")
ejes[1].legend()

plt.tight_layout()
plt.savefig("reports/matriz_confusion.png", dpi=120)
plt.show()

print(f"Negativas confundidas con positivas: {matriz[0, 1]}")
print(f"Positivas confundidas con negativas: {matriz[1, 0]}")

[Escribe aquí el accuracy y el AUC de prueba, y qué tipo de error comete más el modelo según la matriz de confusión.]

## 10. Predicciones de prueba

### 10.1 Reseñas del conjunto de prueba

12 reseñas elegidas al azar que el modelo nunca vio. La tabla muestra el comienzo del texto, la clase real, la predicha y la probabilidad de que sea positiva.

In [ ]:
# Volvemos a cargar las reseñas sin recortar, solo para mostrarlas completas
(_, _), (x_test_texto, _) = keras.datasets.imdb.load_data(num_words=NUM_PALABRAS)

rng = np.random.default_rng(SEMILLA)
indices = rng.choice(len(x_test), size=12, replace=False)

tabla = pd.DataFrame({
    "Reseña (inicio)": [decodificar(x_test_texto[i])[:110] + "..." for i in indices],
    "Real": [CLASES[y_test[i]] for i in indices],
    "Predicha": [CLASES[y_pred[i]] for i in indices],
    "P(positiva)": [round(float(probabilidades[i]), 3) for i in indices],
    "¿Acierto?": ["✔" if y_pred[i] == y_test[i] else "✘" for i in indices],
})

pd.set_option("display.max_colwidth", 120)
display(tabla)
print("Aciertos:", int((y_pred[indices] == y_test[indices]).sum()), "de 12")

### 10.2 Reseñas escritas a mano

Ahora pruebo el camino completo que va a seguir la aplicación: texto escrito por una persona → limpieza → números → red. Incluyo casos fáciles y casos con trampa, como una negación o una opinión mezclada.

In [ ]:
resenas_nuevas = [
    "An absolute masterpiece. The performances are outstanding and the music is beautiful.",
    "Terrible movie. Boring, predictable and way too long. I want my money back.",
    "I didn't like it at all.",
    "Not bad at all, actually quite enjoyable.",
    "The first half was great, but the ending ruined everything.",
    "It was okay. Nothing special, nothing terrible.",
]

filas = []
for texto in resenas_nuevas:
    p = float(modelo.predict(codificar(texto), verbose=0)[0][0])
    filas.append({"Reseña": texto, "Predicha": CLASES[int(p >= 0.5)], "P(positiva)": round(p, 3)})

display(pd.DataFrame(filas))

[Escribe aquí qué pasó con las reseñas difíciles: la negación "Not bad at all", la opinión mezclada y la neutral.]

### 10.3 Errores con mayor confianza

In [ ]:
errores = np.where(y_pred != y_test)[0]
confianza_error = np.abs(probabilidades[errores] - 0.5)
peores = errores[np.argsort(confianza_error)[::-1][:5]]

for i in peores:
    print(f"Real: {CLASES[y_test[i]]} | Predicha: {CLASES[y_pred[i]]} | P(positiva) = {probabilidades[i]:.3f}")
    print("   ", decodificar(x_test_texto[i])[:350], "...")
    print()

print(f"Total de errores: {len(errores)} de {len(x_test)}")

## 11. Guardado del modelo entrenado

Guardo tres archivos. El modelo en formato `.keras`. El vocabulario, porque la aplicación necesita traducir palabras a números exactamente igual que en el entrenamiento. Y una configuración con el largo de secuencia y los nombres de las clases.

In [ ]:
modelo.save("models/lstm_imdb.keras")

with open("models/vocabulario.json", "w", encoding="utf-8") as f:
    json.dump(vocabulario, f, ensure_ascii=False)

with open("models/config.json", "w", encoding="utf-8") as f:
    json.dump({"num_palabras": NUM_PALABRAS, "longitud": LONGITUD, "clases": CLASES},
              f, ensure_ascii=False, indent=2)

for archivo in os.listdir("models"):
    print(f"{archivo:<20} {os.path.getsize(os.path.join('models', archivo)) / 1e6:.2f} MB")

Compruebo que el archivo guardado funciona: lo cargo de nuevo y debe dar el mismo accuracy.

In [ ]:
modelo_cargado = keras.models.load_model("models/lstm_imdb.keras")
_, accuracy_cargado = modelo_cargado.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy del modelo original: {accuracy_test:.4f}")
print(f"Accuracy del modelo cargado:  {accuracy_cargado:.4f}")

### Descargar los resultados de Colab

Colab borra los archivos al cerrar la sesión. Esta celda empaqueta `models/` y `reports/` en un zip y lo descarga. Su contenido va a las carpetas con el mismo nombre en el repositorio.

In [ ]:
import zipfile

with zipfile.ZipFile("resultados_lstm.zip", "w") as zf:
    for carpeta in ["models", "reports"]:
        for archivo in os.listdir(carpeta):
            zf.write(os.path.join(carpeta, archivo))

try:
    from google.colab import files
    files.download("resultados_lstm.zip")
except ImportError:
    print("No estás en Colab: el zip quedó en la carpeta actual.")

## 12. Conclusiones

[Completa con tus resultados reales.]